# Training dei modelli LSTM per seed

Per ogni seed vengono allenati tutti i modelli (`LSTM3y`, `LSTM13y`, `cumulative` e `rolling`) e salvati in `runs/seed_{seed}`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys
from pathlib import Path

# Impostare i seed prima di importare TensorFlow/Keras rende riproducibile
# l'inizializzazione dei pesi e l'ordine casuale dei batch.
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path('/content/drive/MyDrive/master-thesis').resolve()
SRC_PATH = PROJECT_ROOT / 'src'
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(SRC_PATH))

from scr.model.model_training_LSTM import train_and_save_models
from scr.model.cumulative_retraining import train_and_save_cumulative_quarterly_models
from scr.model.rolling_retraining import train_and_save_rolling_quarterly_models

TICKERS = [
    'GOOGL', 'AMZN', 'AAPL', 'PEP', 'JNJ', 'PFE', 'MRK', 'ABBV', 'PG', 'KO',
    'WMT', 'JPM', 'BAC', 'GS', 'V', 'XOM', 'CVX', 'COP', 'BP', 'BA',
    'MMM', 'HON', 'GE', 'T', 'VZ', 'TMUS', 'HSY', 'DUK', 'SO', 'EXC', 'AEP',
    'AMT', 'PLD', 'SPG', 'BHP', 'RIO', 'VALE', 'FCX',
]

# Modificare questa lista per aggiungere o rimuovere seed.
SEEDS = [42, 43, 44]
SEQUENCE_LENGTH = 50
DATA_START = '2013-01-01'
SIMULATION_START = '2021-01-01'
SIMULATION_END = '2026-06-01'

DATA_3Y = PROJECT_ROOT / 'data/3_years'
DATA_13Y = PROJECT_ROOT / 'data/13_years'
RUNS_DIR = PROJECT_ROOT / 'runs'


In [ ]:
training_results = {}

for seed in SEEDS:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass
    np.random.seed(seed)

    run_dir = RUNS_DIR / f'seed_{seed}'
    model_dirs = {
        'LSTM3y': run_dir / 'models/LSTM_3_years',
        'LSTM13y': run_dir / 'models/LSTM_13_years',
        'cumulative': run_dir / 'models/LSTM_cumulative',
        'rolling': run_dir / 'models/LSTM_rolling',
    }
    for model_dir in model_dirs.values():
        model_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n===== seed {seed} =====')
    training_results[seed] = {}

    # Modelli con finestra storica fissa di 3 e 13 anni.
    training_results[seed]['LSTM3y'] = train_and_save_models(
        TICKERS, SEQUENCE_LENGTH, model_folder=str(model_dirs['LSTM3y']),
        data_folder=str(DATA_3Y), force_retrain=True,
    )
    training_results[seed]['LSTM13y'] = train_and_save_models(
        TICKERS, SEQUENCE_LENGTH, model_folder=str(model_dirs['LSTM13y']),
        data_folder=str(DATA_13Y), force_retrain=True,
    )

    # Modelli riaddestrati a ogni trimestre.
    training_results[seed]['cumulative'] = train_and_save_cumulative_quarterly_models(
        TICKERS, SIMULATION_START, SIMULATION_END, SEQUENCE_LENGTH,
        data_start=DATA_START, model_folder=str(model_dirs['cumulative']),
        force_retrain=True,
    )
    training_results[seed]['rolling'] = train_and_save_rolling_quarterly_models(
        TICKERS, SIMULATION_START, SIMULATION_END, SEQUENCE_LENGTH,
        data_start=DATA_START, model_folder=str(model_dirs['rolling']),
        force_retrain=True,
    )

print('\nTraining completato. Modelli salvati in:')
for seed in SEEDS:
    print(f'  {RUNS_DIR / f"seed_{seed}"}')
